# German Cities Project - Web Scraping, APIs, and MySQL

This notebook builds a small database for three German cities - 
Berlin, Hamburg, and Munich - combining data from three different 
sources:

- **Wikipedia** (web scraping): country, coordinates, and population
- **OpenWeather API**: 5-day weather forecast
- **AeroDataBox API** (via RapidAPI): each city's major airport and 
  tomorrow's flight arrivals

All of this is loaded into a MySQL database with five tables. `cities` 
is static reference data, while `populations`, `forecasts`, and 
`arrivals` hold data that changes over time, and `airports` holds 
each city's major airport info. Everything links back to `cities` 
through a shared `city_id`.

The idea behind this structure: Gans (a fictional company in this 
exercise) wants a centralized view of population trends, weather 
patterns, and flight arrivals across cities, to help inform decisions 
like fleet positioning.

API keys and database credentials are stored in a `.env` file (not 
tracked by git) rather than typed directly into this notebook - see 
`.env.example` for the required variables.

- Berlin: https://en.wikipedia.org/wiki/Berlin
- Hamburg: https://en.wikipedia.org/wiki/Hamburg
- Munich: https://en.wikipedia.org/wiki/Munich

 

## 1. Imports and Setup

Importing the libraries needed for web scraping (`requests`, `BeautifulSoup`), 
data handling (`pandas`), and text cleaning (`re`). 

A custom `User-Agent` header is set because some websites, including 
Wikipedia, may block or limit requests that don't look like they're 
coming from a real browser.

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from datetime import datetime

headers = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/152.0.0.0 Safari/537.36"}

## 2. Exploring the Page Structure (Berlin)

Before writing the scraping loop for all three cities, it helps to 
inspect a single page first - Berlin - to understand how the 
information is structured in Wikipedia's infobox. This step is just 
for exploration and isn't part of the final scraping logic.

In [2]:
url = "https://en.wikipedia.org/wiki/Berlin"
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.content, "html.parser")

# The infobox on the right side of most Wikipedia pages has class "infobox"
infobox = soup.find("table", class_="infobox")
print(infobox.prettify()[:2000])  # inspect structure

<table about="#mwt6" class="infobox ib-settlement vcard" id="mwCw">
 <tbody>
  <tr>
   <th class="infobox-above" colspan="2">
    <div class="fn org">
     Berlin
    </div>
   </th>
  </tr>
  <tr>
   <td class="infobox-subheader" colspan="2">
    <div class="category">
     Capital city,
     <a href="https://en.wikipedia.org/wiki/States_of_Germany" rel="mw:WikiLink" title="States of Germany">
      state
     </a>
     and
     <a href="https://en.wikipedia.org/wiki/Municipalities_of_Germany" rel="mw:WikiLink" title="Municipalities of Germany">
      municipality
     </a>
    </div>
   </td>
  </tr>
  <tr class="mergedtoprow">
   <td class="infobox-full-data" colspan="2">
    <span class="mw-empty-elt">
     <style about="#mwt9" data-mw='{"name":"templatestyles","attrs":{"src":"Multiple image/styles.css","wrapper":".tmulti"},"body":{"extsrc":""}}' data-mw-deduplicate="TemplateStyles:r1349637415/mw-parser-output/.tmulti" typeof="mw:Extension/templatestyles">
      .mw-parser-output .

## 3. Extracting Country and Coordinates

Looking at Berlin's infobox, the country appears in a row where the 
`<th>` cell contains the word "Country" - its value sits in the 
neighboring `<td>`. Coordinates are stored separately, in a 
`<span class="geo-dec">` element near the top of the page, formatted 
as a string like `"52.520008°N 13.404954°E"`.

Testing this on Hamburg and Munich showed the same structure works 
for all three cities - only the URL changes. That repetition is the 
signal to turn this into a loop instead of repeating the same code 
three times.

In [3]:
urls = {
    "Berlin": "https://en.wikipedia.org/wiki/Berlin",
    "Hamburg": "https://en.wikipedia.org/wiki/Hamburg",
    "Munich": "https://en.wikipedia.org/wiki/Munich",
}

data = []

for city, url in urls.items():
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    infobox = soup.find("table", class_="infobox")

    # Country
    country = None
    for row in infobox.find_all("tr"):
        header = row.find("th")
        if header and "Country" in header.text:
            country = row.find("td").get_text(strip=True)
            break

    # Coordinates (raw string, decimal degrees)
    geo = soup.find("span", class_="geo-dec")
    geo_text = geo.text if geo else None

    data.append({
        "city": city,
        "country": country,
        "coordinates_raw": geo_text
    })

data

[{'city': 'Berlin',
  'country': 'Germany',
  'coordinates_raw': '52.52000°N 13.40500°E'},
 {'city': 'Hamburg',
  'country': 'Germany',
  'coordinates_raw': '53.550°N 10.000°E'},
 {'city': 'Munich',
  'country': 'Germany',
  'coordinates_raw': '48.13750°N 11.57500°E'}]

### Converting Coordinates to Decimal Format

The scraped coordinate string, e.g. `"52.520008°N 13.404954°E"`, isn't 
usable as numbers yet. This function splits it into latitude and 
longitude, removes the degree symbol and direction letter, and 
converts South/West values to negative - giving clean decimal 
coordinates.

In [4]:
def parse_coordinates(geo_text):
    lat_str, lon_str = geo_text.split(" ")

    lat_val = float(re.sub(r"[°NS]", "", lat_str))
    if "S" in lat_str:
        lat_val = -lat_val

    lon_val = float(re.sub(r"[°EW]", "", lon_str))
    if "W" in lon_str:
        lon_val = -lon_val

    return lat_val, lon_val

# test
parse_coordinates("52.520008°N 13.404954°E")
# -> (52.520008, 13.404954)

(52.520008, 13.404954)

### Applying the Function and Building a DataFrame

With `parse_coordinates()` ready, we can convert every city's raw 
coordinate string into proper decimal latitude/longitude values, and 
organize everything into a pandas DataFrame.

In [5]:
for row in data:
    lat, lon = parse_coordinates(row["coordinates_raw"])
    row["latitude"] = lat
    row["longitude"] = lon
    del row["coordinates_raw"]

df = pd.DataFrame(data)
df = df[["city", "country", "latitude", "longitude"]]
df

,city,country,latitude,longitude
0,Berlin,Germany,52.5200,13.405
1,Hamburg,Germany,53.5500,10.000
2,Munich,Germany,48.1375,11.575


### Wrapping the Process into a Reusable Function

To make this scraping process reusable - for example, if more cities 
are added later - the logic above is wrapped into a single function, 
`get_city_geodata()`. It takes a dictionary of `{city_name: url}` and 
returns a clean DataFrame with country, latitude, and longitude for 
each one.

In [6]:
def get_city_geodata(city_urls: dict) -> pd.DataFrame:
    """
    Scrape country, latitude, and longitude for each city
    from its Wikipedia page.

    Parameters
    ----------
    city_urls : dict
        Mapping of {city_name: wikipedia_url}

    Returns
    -------
    pd.DataFrame with columns: city, country, latitude, longitude
    """
    headers = {"User-Agent": "Mozilla/5.0 (WBS Coding School Project)"}
    records = []

    for city, url in city_urls.items():
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, "html.parser")
        infobox = soup.find("table", class_="infobox")

        # Country
        country = None
        if infobox:
            for row in infobox.find_all("tr"):
                header = row.find("th")
                if header and "Country" in header.text:
                    country = row.find("td").get_text(strip=True)
                    break

        # Coordinates
        geo = soup.find("span", class_="geo-dec")
        if geo:
            lat, lon = parse_coordinates(geo.text)
        else:
            lat, lon = None, None

        records.append({
            "city": city,
            "country": country,
            "latitude": lat,
            "longitude": lon
        })

    return pd.DataFrame(records)[["city", "country", "latitude", "longitude"]]


# Usage:
urls = {
    "Berlin": "https://en.wikipedia.org/wiki/Berlin",
    "Hamburg": "https://en.wikipedia.org/wiki/Hamburg",
    "Munich": "https://en.wikipedia.org/wiki/Munich",
}
df = get_city_geodata(urls)
df

,city,country,latitude,longitude
0,Berlin,Germany,52.5200,13.405
1,Hamburg,Germany,53.5500,10.000
2,Munich,Germany,48.1375,11.575


## 4. Extracting Population

Wikipedia's infobox doesn't label population with a simple, consistent 
header - it often includes the census year and footnote markers (e.g. 
"Population(2022 census)[2]"). This function searches for any header 
containing the word "Population", then reads the number from the row 
right below it, cleaning out commas and footnote references.

Since population changes over time, we don't store this number alone - 
later, when we insert it into MySQL, we'll pair it with a timestamp 
marking exactly when it was scraped. This lets us keep a history of 
population values instead of overwriting old data.

In [7]:
import re

def get_population(infobox):
    """
    Extract a city's population from its Wikipedia infobox.
    Searches for a header row containing 'Population', then reads 
    the number from the row directly below it.
    """
    rows = infobox.find_all("tr")

    for i, row in enumerate(rows):
        header = row.find("th")
        if header and "Population" in header.get_text(strip=True):
            # the actual number usually sits in the next row's <td>
            next_row = rows[i + 1]
            td = next_row.find("td")
            if td:
                raw_text = td.get_text(strip=True)
                # remove footnote markers like [2], then strip everything 
                # that isn't a digit (commas, spaces, etc.)
                clean_number = re.sub(r"[^\d]", "", raw_text.split("[")[0])
                if clean_number:
                    return int(clean_number)
    return None  # in case population isn't found, avoids crashing the loop

## 5. Putting It All Together - Final Scrape

The sections above explored each piece separately: extracting country 
and coordinates, converting coordinates to decimal format, and 
extracting population. This final loop combines all of that into a 
single pass over the three cities, producing one complete DataFrame - 
with country, latitude, longitude, and population - that is used for 
all the database steps below.

In [8]:
urls = {
    "Berlin": "https://en.wikipedia.org/wiki/Berlin",
    "Hamburg": "https://en.wikipedia.org/wiki/Hamburg",
    "Munich": "https://en.wikipedia.org/wiki/Munich",
}

data = []

for city, url in urls.items():
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    infobox = soup.find("table", class_="infobox")

    # --- Country ---
    country = None
    for row in infobox.find_all("tr"):
        header = row.find("th")
        if header and "Country" in header.text:
            country = row.find("td").get_text(strip=True)
            break

    # --- Coordinates ---
    geo = soup.find("span", class_="geo-dec")
    geo_text = geo.text if geo else None
    lat, lon = parse_coordinates(geo_text) if geo_text else (None, None)

    # --- Population ---
    population = get_population(infobox)

    data.append({
        "city": city,
        "country": country,
        "latitude": lat,
        "longitude": lon,
        "population": population
    })

df = pd.DataFrame(data)
df

,city,country,latitude,longitude,population
0,Berlin,Germany,52.5200,13.405,3596999
1,Hamburg,Germany,53.5500,10.000,1973896
2,Munich,Germany,48.1375,11.575,1505036


## 6. Database Connection Setup

Credentials (host, user, password, API keys, etc.) are stored in a `.env` file, 
which is excluded from GitHub via `.gitignore` so no sensitive 
information is ever exposed. A safe placeholder version, `.env.example`, 
is included in the repo so anyone cloning this project knows which 
variables they need to set up themselves.

In [9]:
!pip install pymysql sqlalchemy

In [10]:
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine, text

load_dotenv()  # reads the .env file and loads credentials into memory

host = os.getenv("DB_HOST")
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
port = os.getenv("DB_PORT")
schema = os.getenv("DB_SCHEMA")

connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'
engine = create_engine(connection_string)  # this is our live connection to MySQL

## 7. Creating Tables

This cell resets every table used later in the notebook (forecasts, 
arrivals, airports, populations, cities), even though only cities 
and populations are created right here - this keeps the drop-and-
recreate logic in one place, so re-running the whole notebook always 
starts from a clean slate no matter which sections run after it. 
forecasts, arrivals, and airports get created later, in their own 
sections, once their data is ready.

Creating two tables here: cities (static city info) and populations 
(population values over time, linked to cities via city_id as a 
foreign key).

In [11]:
from sqlalchemy import text
# Reset tables so re-running this notebook never creates duplicates.
# forecasts must be dropped before cities, since forecasts has a
# foreign key pointing back to cities - MySQL won't let you drop
# cities while something still references it
with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS forecasts"))
    conn.execute(text("DROP TABLE IF EXISTS arrivals"))
    conn.execute(text("DROP TABLE IF EXISTS airports"))
    conn.execute(text("DROP TABLE IF EXISTS populations"))
    conn.execute(text("DROP TABLE IF EXISTS cities"))
    conn.commit()


create_cities_table = """
CREATE TABLE cities (
    city_id INT PRIMARY KEY,
    city_name VARCHAR(255) NOT NULL,
    country VARCHAR(255) NOT NULL,
    latitude DECIMAL(9,6),
    longitude DECIMAL(9,6)
);
"""

create_populations_table = """
CREATE TABLE populations (
    population_id INT PRIMARY KEY AUTO_INCREMENT,
    city_id INT NOT NULL,
    population INT NOT NULL,
    timestamp DATETIME NOT NULL,
    FOREIGN KEY (city_id) REFERENCES cities(city_id)
);
"""

with engine.connect() as conn:
    conn.execute(text(create_cities_table))
    conn.execute(text(create_populations_table))
    conn.commit()

In [12]:
pd.read_sql("SHOW TABLES", engine)

,Tables_in_sql_workshop
0,cities
1,populations


## 8. Inserting Data

Inserting the scraped city data into `cities`, and the scraped 
population figures into `populations`. Each population entry is 
stamped with the current date and time, so future re-scrapes can be 
added as new rows without overwriting this history.

In [13]:
#inserting into cities

for i, row in df.iterrows():
    insert_query = text("""
        INSERT INTO cities (city_id, city_name, country, latitude, longitude)
        VALUES (:city_id, :city_name, :country, :latitude, :longitude)
    """)
    with engine.connect() as conn:
        conn.execute(insert_query, {
            "city_id": i + 1,          # Berlin=1, Hamburg=2, Munich=3
            "city_name": row["city"],
            "country": row["country"],
            "latitude": row["latitude"],
            "longitude": row["longitude"]
        })
        conn.commit()

In [14]:
# insert into populations

for i, row in df.iterrows():
    insert_query = text("""
        INSERT INTO populations (city_id, population, timestamp)
        VALUES (:city_id, :population, :timestamp)
    """)
    with engine.connect() as conn:
        conn.execute(insert_query, {
            "city_id": i + 1,
            "population": row["population"],
            "timestamp": datetime.now()
        })
        conn.commit()

In [15]:
print("Cities table:")
display(pd.read_sql("SELECT * FROM cities", engine))

print("\nPopulations table:")
display(pd.read_sql("SELECT * FROM populations", engine))

Cities table:


,city_id,city_name,country,latitude,longitude
0,1,Berlin,Germany,52.5200,13.405
1,2,Hamburg,Germany,53.5500,10.000
2,3,Munich,Germany,48.1375,11.575



Populations table:


,population_id,city_id,population,timestamp
0,1,1,3596999,2026-09-24 17:33:18
1,2,2,1973896,2026-09-24 17:33:18
2,3,3,1505036,2026-09-24 17:33:18


## 9. Weather Forecast - Exploring the API

For this section, an OpenWeather API key is required. Following the 
same approach used for the MySQL credentials, the key is stored in 
the existing `.env` file (as `OPENWEATHER_API_KEY`) rather than being 
typed directly into this notebook, so it stays out of the published 
version on GitHub. The `.env.example` file has been updated to reflect 
this new variable as well.

Using the latitude and longitude already scraped from Wikipedia, we 
first explore the response from OpenWeather's 5-day forecast API for 
a single city (Berlin), to understand its structure before automating 
collection for all cities.

In [16]:
# Using Berlin's coordinates from our scraped df
API_KEY = os.getenv("OPENWEATHER_API_KEY")
berlin_lat = df.loc[df["city"] == "Berlin", "latitude"].values[0]
berlin_lon = df.loc[df["city"] == "Berlin", "longitude"].values[0]

url = f"https://api.openweathermap.org/data/2.5/forecast?lat={berlin_lat}&lon={berlin_lon}&appid={API_KEY}&units=metric"

response = requests.get(url)
weather_data = response.json()

weather_data

{'cod': '200',
 'message': 0,
 'cnt': 40,
 'list': [{'dt': 1790272800,
   'main': {'temp': 14.38,
    'feels_like': 14.09,
    'temp_min': 14.38,
    'temp_max': 15.64,
    'pressure': 1019,
    'sea_level': 1019,
    'grnd_level': 1015,
    'humidity': 85,
    'temp_kf': -1.26,
    'dew_point': 11.89},
   'weather': [{'id': 803,
     'main': 'Clouds',
     'description': 'broken clouds',
     'icon': '04n'}],
   'clouds': {'all': 84},
   'wind': {'speed': 2.94, 'deg': 285, 'gust': 6.08},
   'visibility': 10000,
   'pop': 0,
   'sys': {'pod': 'n'},
   'dt_txt': '2026-09-24 18:00:00'},
  {'dt': 1790283600,
   'main': {'temp': 14.56,
    'feels_like': 14.29,
    'temp_min': 14.56,
    'temp_max': 14.97,
    'pressure': 1020,
    'sea_level': 1020,
    'grnd_level': 1016,
    'humidity': 85,
    'temp_kf': -0.41,
    'dew_point': 12.07},
   'weather': [{'id': 804,
     'main': 'Clouds',
     'description': 'overcast clouds',
     'icon': '04n'}],
   'clouds': {'all': 92},
   'wind': {'spe

### Exploring the API (Berlin only)

In [17]:
# See the top-level keys first
print(weather_data.keys())

dict_keys(['cod', 'message', 'cnt', 'list', 'city'])


In [18]:
# Look at just the first forecast entry
weather_data["list"][0]

{'dt': 1790272800,
 'main': {'temp': 14.38,
  'feels_like': 14.09,
  'temp_min': 14.38,
  'temp_max': 15.64,
  'pressure': 1019,
  'sea_level': 1019,
  'grnd_level': 1015,
  'humidity': 85,
  'temp_kf': -1.26,
  'dew_point': 11.89},
 'weather': [{'id': 803,
   'main': 'Clouds',
   'description': 'broken clouds',
   'icon': '04n'}],
 'clouds': {'all': 84},
 'wind': {'speed': 2.94, 'deg': 285, 'gust': 6.08},
 'visibility': 10000,
 'pop': 0,
 'sys': {'pod': 'n'},
 'dt_txt': '2026-09-24 18:00:00'}

In [19]:
# How many forecast entries are there in total?
len(weather_data["list"])

40

### Extracting Useful Fields into a DataFrame

From each forecast entry, we keep the fields that are genuinely useful 
for a general weather overview: date/time, temperature, humidity, 
weather description, wind speed, and chance of rain. Fields like 
pressure, dew point, and visibility are dropped since they add noise 
without much practical value for this project.

In [20]:
forecast_list = weather_data["list"]

records = []
for entry in forecast_list:
    records.append({
        "datetime": entry["dt_txt"],
        "temperature": entry["main"]["temp"],
        "humidity": entry["main"]["humidity"],
        "description": entry["weather"][0]["description"],
        "wind_speed": entry["wind"]["speed"],
        "chance_of_rain": entry["pop"]
    })

berlin_weather_df = pd.DataFrame(records)
berlin_weather_df

,datetime,temperature,humidity,description,wind_speed,chance_of_rain
0,2026-09-24 18:00:00,14.38,85,broken clouds,2.94,0.00
1,2026-09-24 21:00:00,14.56,85,overcast clouds,3.08,0.00
2,2026-09-25 00:00:00,14.56,90,overcast clouds,2.66,0.00
3,2026-09-25 03:00:00,14.74,89,light rain,2.87,0.93
4,2026-09-25 06:00:00,11.74,93,light rain,2.86,0.60
5,2026-09-25 09:00:00,16.83,73,broken clouds,3.80,0.00
6,2026-09-25 12:00:00,16.97,62,overcast clouds,3.63,0.00
7,2026-09-25 15:00:00,17.01,56,scattered clouds,2.62,0.00
8,2026-09-25 18:00:00,15.57,75,clear sky,1.71,0.00
9,2026-09-25 21:00:00,12.70,82,clear sky,1.99,0.00


## 10. Automating Weather Collection for Multiple Cities

Instead of using the cities dataframe still sitting in memory from the 
scraping step, I'm pulling latitude and longitude directly from the 
cities table in MySQL - this way the weather step doesn't depend on 
having already run the scraping code earlier in the same session.

Added status code check used for the airports function - if 
a request fails for a city, it prints a clear warning instead of 
silently skipping it.

In [21]:
# re-reading .env here specifically, so this section doesn't depend
# on load_dotenv() having already run earlier in the notebook
load_dotenv()
API_KEY = os.getenv("OPENWEATHER_API_KEY")

def get_weather_forecast(engine) -> pd.DataFrame:
    """
    Pulls the cities straight from mysql, then gets the 5-day forecast
    for each one from OpenWeather. Returns one combined dataframe.
    """

    cities_df = pd.read_sql("SELECT * FROM cities", con=engine)

    # turning the columns into plain lists so I can loop through them together
    city_names = cities_df["city_name"].to_list()
    latitudes = cities_df["latitude"].to_list()
    longitudes = cities_df["longitude"].to_list()

    all_records = []

    # zip() lets me loop through all three lists at once, matching up
    # by position - city_names[0] goes with latitudes[0] and longitudes[0], etc
    for city, lat, lon in zip(city_names, latitudes, longitudes):
        url = f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={API_KEY}&units=metric"
        response = requests.get(url)

        # checking the status code instead of assuming it worked - if
        # something's wrong (bad key, rate limit) I get a clear message
        # instead of it silently breaking or giving a confusing error later
        if response.status_code == 200:
            weather_data = response.json()

            # weather_data["list"] has 40 entries per city (every 3 hours, for 5 days)
            for entry in weather_data["list"]:
                all_records.append({
                    "city": city,  # tagging each row so I know which city it belongs to
                    "datetime": entry["dt_txt"],
                    "temperature": entry["main"]["temp"],
                    "humidity": entry["main"]["humidity"],
                    "description": entry["weather"][0]["description"],
                    "wind_speed": entry["wind"]["speed"],
                    "chance_of_rain": entry["pop"]
                })
        else:
            print(f"⚠️ request failed for {city} - status {response.status_code}: {response.text}")

    if not all_records:
        # nothing came back for any city - raise a clear error here
        # instead of returning an empty/broken dataframe silently
        raise ValueError("No weather data was returned for any city - check the warnings above")

    return pd.DataFrame(all_records)

### Applying the Function to Our Scraped Cities

Using `get_weather_forecast()` with the `df` created earlier from the 
Wikipedia scraping step, giving us weather forecasts for Berlin, 
Hamburg, and Munich in one call.

In [22]:
weather_df = get_weather_forecast(engine)
weather_df

,city,datetime,temperature,humidity,description,wind_speed,chance_of_rain
0,Berlin,2026-09-24 18:00:00,14.38,85,broken clouds,2.94,0.00
1,Berlin,2026-09-24 21:00:00,14.56,85,overcast clouds,3.08,0.00
2,Berlin,2026-09-25 00:00:00,14.56,90,overcast clouds,2.66,0.00
3,Berlin,2026-09-25 03:00:00,14.74,89,light rain,2.87,0.93
4,Berlin,2026-09-25 06:00:00,11.74,93,light rain,2.86,0.60
...,...,...,...,...,...,...,...
115,Munich,2026-09-29 03:00:00,11.80,90,clear sky,2.08,0.00
116,Munich,2026-09-29 06:00:00,11.11,94,clear sky,1.18,0.00
117,Munich,2026-09-29 09:00:00,17.33,68,few clouds,0.92,0.00
118,Munich,2026-09-29 12:00:00,22.71,53,clear sky,1.74,0.00


In [23]:
weather_df["city"].value_counts()

city
Berlin     40
Hamburg    40
Munich     40
Name: count, dtype: int64

## 11. Loading Weather Data into MySQL

The forecasts table needs a city_id to match the pattern used for 
populations, but weather_df only has the city name right now. So 
first I need to pull city_id back from the cities table already in 
MySQL and match it up with the right city name.

In [24]:
# grabbing city_id + city_name from the cities table already in mysql,
# so I can match it up with my weather_df (which only has city names right now)
cities_from_sql = pd.read_sql("SELECT city_id, city_name FROM cities", engine)

# merging weather_df with cities_from_sql - matches rows where
# weather_df's "city" text equals cities_from_sql's "city_name" text
weather_df_merged = weather_df.merge(
    cities_from_sql,
    left_on="city",
    right_on="city_name"
)

# don't need the text columns anymore now that I have city_id
weather_df_merged = weather_df_merged.drop(columns=["city", "city_name"])
weather_df_merged

,datetime,temperature,humidity,description,wind_speed,chance_of_rain,city_id
0,2026-09-24 18:00:00,14.38,85,broken clouds,2.94,0.00,1
1,2026-09-24 21:00:00,14.56,85,overcast clouds,3.08,0.00,1
2,2026-09-25 00:00:00,14.56,90,overcast clouds,2.66,0.00,1
3,2026-09-25 03:00:00,14.74,89,light rain,2.87,0.93,1
4,2026-09-25 06:00:00,11.74,93,light rain,2.86,0.60,1
...,...,...,...,...,...,...,...
115,2026-09-29 03:00:00,11.80,90,clear sky,2.08,0.00,3
116,2026-09-29 06:00:00,11.11,94,clear sky,1.18,0.00,3
117,2026-09-29 09:00:00,17.33,68,few clouds,0.92,0.00,3
118,2026-09-29 12:00:00,22.71,53,clear sky,1.74,0.00,3


### Creating the Forecasts Table

Same pattern as populations - dropping and recreating the table each 
time so re-running this notebook never causes duplicates. city_id 
links back to cities, just like before.

In [25]:
# dropping first so re-running this notebook always starts fresh, no duplicates
with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS forecasts"))
    conn.commit()

create_forecasts_table = """
CREATE TABLE forecasts (
    forecast_id INT PRIMARY KEY AUTO_INCREMENT,
    city_id INT NOT NULL,
    datetime DATETIME NOT NULL,
    temperature DECIMAL(5,2),
    humidity INT,
    description VARCHAR(255),
    wind_speed DECIMAL(5,2),
    chance_of_rain DECIMAL(3,2),
    FOREIGN KEY (city_id) REFERENCES cities(city_id)
);
"""

with engine.connect() as conn:
    conn.execute(text(create_forecasts_table))
    conn.commit()

### Inserting with .to_sql()

Instead of looping through rows and writing INSERT statements myself 
(like I did for cities and populations), pandas can push the whole 
dataframe into the table in one line with .to_sql().

In [26]:
# name = table to insert into
# con = my db connection
# if_exists="append" = add these rows without wiping the table
#   (safe here since I already drop + recreate the table above every run)
# index=False = don't include pandas' own row numbers as a column
weather_df_merged.to_sql(
    name="forecasts",
    con=engine,
    if_exists="append",
    index=False
)

120

### Checking It Worked

Pulling the data back out of mysql to make sure everything landed 
correctly - should be 40 rows per city, 120 total.

In [27]:
# quick peek at the table
display(pd.read_sql("SELECT * FROM forecasts LIMIT 10", engine))

# counting rows per city - should be 40 each (1=Berlin, 2=Hamburg, 3=Munich)
display(pd.read_sql("SELECT city_id, COUNT(*) AS row_count FROM forecasts GROUP BY city_id", engine))

,forecast_id,city_id,datetime,temperature,humidity,description,wind_speed,chance_of_rain
0,1,1,2026-09-24 18:00:00,14.38,85,broken clouds,2.94,0.00
1,2,1,2026-09-24 21:00:00,14.56,85,overcast clouds,3.08,0.00
2,3,1,2026-09-25 00:00:00,14.56,90,overcast clouds,2.66,0.00
3,4,1,2026-09-25 03:00:00,14.74,89,light rain,2.87,0.93
4,5,1,2026-09-25 06:00:00,11.74,93,light rain,2.86,0.60
5,6,1,2026-09-25 09:00:00,16.83,73,broken clouds,3.80,0.00
6,7,1,2026-09-25 12:00:00,16.97,62,overcast clouds,3.63,0.00
7,8,1,2026-09-25 15:00:00,17.01,56,scattered clouds,2.62,0.00
8,9,1,2026-09-25 18:00:00,15.57,75,clear sky,1.71,0.00
9,10,1,2026-09-25 21:00:00,12.70,82,clear sky,1.99,0.00


,city_id,row_count
0,1,40
1,2,40
2,3,40


## 12. Flights - Finding Each City's Airport

### Adding the AeroDataBox API Key

An AeroDataBox API key (via RapidAPI) is stored in .env as 
RAPIDAPI_KEY, same approach as the other credentials. .env.example 
has been updated to show this variable too.

### Searching for Airports Near Each City

Using the AeroDataBox "Search airports by location" endpoint to find 
the nearest major airport to each city, based on latitude/longitude. 
Pulling city data from mysql (same as the weather step), so this 
doesn't depend on anything still sitting in memory.

In [28]:
# re-reading .env here specifically, so this section doesn't depend
# on load_dotenv() having already run earlier in the notebook
load_dotenv()
RAPIDAPI_KEY = os.getenv("RAPIDAPI_KEY")

def get_airports(engine) -> pd.DataFrame:
    """
    Pulls cities from mysql, then finds the nearest major airport
    (with real flight data) for each one using AeroDataBox.
    """
    cities_df = pd.read_sql("SELECT * FROM cities", con=engine)

    headers = {
        "x-rapidapi-key": RAPIDAPI_KEY,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }

    all_airports = []

    for _, row in cities_df.iterrows():
        url = "https://aerodatabox.p.rapidapi.com/airports/search/location"
        querystring = {
            "lat": row["latitude"],
            "lon": row["longitude"],
            "radiusKm": "50",
            "limit": "10",
            "withFlightInfoOnly": "true"
        }

        response = requests.get(url, headers=headers, params=querystring)

        # checking the status code instead of assuming it worked -
        # this way if something's wrong (bad key, expired subscription,
        # rate limit) I get a clear message instead of a silent skip
        if response.status_code == 200:
            data = response.json()
            airports = pd.json_normalize(data.get("items", []))
            airports["city_id"] = row["city_id"]
            airports["city"] = row["city_name"]
            all_airports.append(airports)
        else:
            print(f"⚠️ request failed for {row['city_name']} - status {response.status_code}: {response.text}")

    if not all_airports:
        # nothing came back at all - raising an actual error here instead
        # of letting pd.concat() fail later with a confusing message
        raise ValueError("No airport data was returned for any city - check the warnings above")

    return pd.concat(all_airports, ignore_index=True)

In [29]:
airports_df = get_airports(engine)
airports_df

,icao,iata,name,shortName,municipalityName,countryCode,timeZone,location.lat,location.lon,city_id,city
0,EDDT,TXL,Berlin -Tegel,-Tegel,Berlin,DE,Europe/Berlin,52.55970,13.287699,1,Berlin
1,EDDB,BER,Berlin Brandenburg,Brandenburg,Berlin,DE,Europe/Berlin,52.35139,13.493889,1,Berlin
2,EDDH,HAM,Hamburg,Hamburg,Hamburg,DE,Europe/Berlin,53.63040,9.988229,2,Hamburg
3,EDDM,MUC,Munich,Munich,Munich,DE,Europe/Berlin,48.35380,11.786100,3,Munich


### Picking the Major Airport per City

Some cities returned more than one airport (Berlin had two). To keep 
this generic and reusable for other cities, I check whether the 
airport's IATA code appears inside the city's name (works for most 
cities, e.g. "HAM" is inside "Hamburg"). Where that doesn't match 
anything (like Munich, whose code MUC comes from the German name 
München), it falls back to keeping the first result for that city, 
and prints a note so it's clear a manual check might be needed.

In [30]:
def pick_major_airport(airports_df: pd.DataFrame) -> pd.DataFrame:
    """
    Keeps one airport per city. Tries to match the airport's IATA code
    against the city name first - if that doesn't find anything for a
    city, falls back to just keeping the first result and prints a note,
    since IATA codes don't always spell out the English city name
    (e.g. Munich's MUC comes from München).
    """
    selected_rows = []

    # going through each city separately, since a city might have
    # more than one candidate airport
    for city_id, group in airports_df.groupby("city_id"):
        city_name = group["city"].iloc[0]

        # check if the iata code shows up inside the city name
        # (case-insensitive) - e.g. "ham" is inside "hamburg"
        match = group[group["iata"].str.lower().apply(lambda code: code in city_name.lower())]

        if len(match) >= 1:
            # take the first match if there's one (or more than one, rare)
            selected_rows.append(match.iloc[0])
        else:
            # no automatic match found - fall back to the first result,
            # but flag it clearly so it's easy to spot and double check
            print(f"⚠️ no iata match found for {city_name} - defaulting to first result ({group.iloc[0]['iata']}). Worth double checking this is the right airport.")
            selected_rows.append(group.iloc[0])

    return pd.DataFrame(selected_rows).reset_index(drop=True)

In [31]:
major_airports_df = pick_major_airport(airports_df)
major_airports_df

⚠️ no iata match found for Munich - defaulting to first result (MUC). Worth double checking this is the right airport.


,icao,iata,name,shortName,municipalityName,countryCode,timeZone,location.lat,location.lon,city_id,city
0,EDDB,BER,Berlin Brandenburg,Brandenburg,Berlin,DE,Europe/Berlin,52.35139,13.493889,1,Berlin
1,EDDH,HAM,Hamburg,Hamburg,Hamburg,DE,Europe/Berlin,53.63040,9.988229,2,Hamburg
2,EDDM,MUC,Munich,Munich,Munich,DE,Europe/Berlin,48.35380,11.786100,3,Munich


### Exploring the Arrivals API

Before building a function for all airports, exploring what a single 
day's arrivals response looks like for one airport (Berlin/EDDB). 

In [32]:
from datetime import date, timedelta

# tomorrow's date, formatted as YYYY-MM-DD for the API
tomorrow = (date.today() + timedelta(days=1)).strftime("%Y-%m-%d")
print(tomorrow)

2026-09-25


In [33]:
# AeroDataBox limits each request to a 12-hour window, so tomorrow's
# full day needs two calls: 00:00-12:00 and 12:00-00:00

url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/EDDB/{tomorrow}T00:00/{tomorrow}T12:00"

querystring = {
    "direction": "Arrival",
    "withLeg": "false",
    "withCancelled": "false",
    "withCodeshared": "true",
    "withCargo": "false",
    "withPrivate": "false"
}

headers = {
    "x-rapidapi-key": RAPIDAPI_KEY,
    "x-rapidapi-host": "aerodatabox.p.rapidapi.com"
}

response = requests.get(url, headers=headers, params=querystring)
print(response.status_code)
print(response.text[:1000])

200
{"arrivals":[{"movement":{"airport":{"icao":"LTBJ","iata":"ADB","name":"İzmir","countryCode":"tr","timeZone":"Europe/Istanbul"},"scheduledTime":{"utc":"2026-09-25 04:05Z","local":"2026-09-25 06:05+02:00"},"revisedTime":{"utc":"2026-09-25 04:05Z","local":"2026-09-25 06:05+02:00"},"terminal":"1","gate":"N01","baggageBelt":"B4","quality":["Basic","Live"]},"number":"XQ 966","status":"Expected","codeshareStatus":"IsOperator","isCargo":false,"aircraft":{"model":"Boeing 737 MAX 8"},"airline":{"name":"Sun Express","iata":"XQ","icao":"SXS"}},{"movement":{"airport":{"icao":"ZBAA","iata":"PEK","name":"Beijing","countryCode":"cn","timeZone":"Asia/Shanghai"},"scheduledTime":{"utc":"2026-09-25 04:45Z","local":"2026-09-25 06:45+02:00"},"revisedTime":{"utc":"2026-09-25 04:45Z","local":"2026-09-25 06:45+02:00"},"terminal":"1","gate":"Y07","baggageBelt":"B3","quality":["Basic","Live"]},"number":"HU 489","status":"Expected","codeshareStatus":"IsOperator","isCargo":false,"aircraft":{"model":"Airbus A3

### Extracting Useful Fields into a DataFrame

From each arrival entry, keeping the fields that actually matter for 
me: where the flight is coming from, when it's scheduled to land, 
the revised/actual time (in case of delays), flight number, airline, 
and status. Skipping the more granular stuff like terminal, gate, 
baggage belt, and aircraft model.

Note: revisedTime only shows up in the API response if a flight's 
time actually changed - for on-time flights it's missing entirely, 
so using .get() with scheduledTime as a fallback instead of assuming 
it's always there.

In [34]:
data = response.json()
arrivals_list = data["arrivals"]

records = []
for entry in arrivals_list:
    movement = entry["movement"]
    
    # revisedTime is only present if the flight's time actually changed -
    # using .get() so this doesn't break on flights that are still on schedule.
    # falling back to scheduledTime when there's no revision
    scheduled = movement["scheduledTime"]["local"]
    revised = movement.get("revisedTime", {}).get("local", scheduled)

    records.append({
        "origin": movement["airport"]["name"],
        "scheduled_arrival": scheduled,
        "actual_arrival": revised,
        "flight_number": entry["number"],
        "airline": entry["airline"]["name"],
        "status": entry["status"]
    })

berlin_arrivals_df = pd.DataFrame(records)
berlin_arrivals_df

,origin,scheduled_arrival,actual_arrival,flight_number,airline,status
0,İzmir,2026-09-25 06:05+02:00,2026-09-25 06:05+02:00,XQ 966,Sun Express,Expected
1,Beijing,2026-09-25 06:45+02:00,2026-09-25 06:45+02:00,HU 489,Hainan,Expected
2,Gaziantep,2026-09-25 06:45+02:00,2026-09-25 06:45+02:00,XQ 1766,Sun Express,Expected
3,Bucharest,2026-09-25 07:05+02:00,2026-09-25 07:05+02:00,W4 3109,Aero Services Executive,Expected
4,Bratislava,2026-09-25 07:15+02:00,2026-09-25 07:15+02:00,W6 7037,Wizz Air,Expected
...,...,...,...,...,...,...
161,Paris,2026-09-25 11:55+02:00,2026-09-25 11:55+02:00,KE 6355,Korean Air,Expected
162,Paris,2026-09-25 11:55+02:00,2026-09-25 11:55+02:00,AF 1734,Air France,Expected
163,Paris,2026-09-25 11:55+02:00,2026-09-25 11:55+02:00,KQ 3230,Kenya,Expected
164,Paris,2026-09-25 11:55+02:00,2026-09-25 11:55+02:00,SV 6320,Saudi Arabian,Expected


### Covering the Full Day

AeroDataBox only allows a 12-hour window per request, so getting a 
full day's arrivals means two calls - 00:00 to 12:00, and 12:00 to 
00:00 (midnight). Combining both into one dataframe for the airport.

In [35]:
def get_arrivals_for_airport(icao: str, day: str) -> pd.DataFrame:
    """
    Gets a full day's arrivals for one airport (by icao code), by
    combining two 12-hour API calls, since AeroDataBox only allows
    a 12-hour window per request.
    """
    headers = {
        "x-rapidapi-key": RAPIDAPI_KEY,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com"
    }

    querystring = {
        "direction": "Arrival",
        "withLeg": "false",
        "withCancelled": "false",
        "withCodeshared": "true",
        "withCargo": "false",
        "withPrivate": "false"
    }

    # two time windows needed to cover the full 24 hours
    time_windows = [
        (f"{day}T00:00", f"{day}T12:00"),
        (f"{day}T12:00", f"{day}T23:59")
    ]

    records = []

    for start, end in time_windows:
        url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/{start}/{end}"
        response = requests.get(url, headers=headers, params=querystring)

        # same safeguard as the other api functions - print a clear
        # warning instead of silently skipping if something goes wrong
        if response.status_code == 200:
            data = response.json()
            arrivals_list = data.get("arrivals", [])

            for entry in arrivals_list:
                movement = entry["movement"]
                scheduled = movement["scheduledTime"]["local"]
                # revisedTime only exists if the flight's time actually
                # changed - falling back to scheduled when it's missing
                revised = movement.get("revisedTime", {}).get("local", scheduled)

                records.append({
                    "icao": icao,
                    "origin": movement["airport"]["name"],
                    "scheduled_arrival": scheduled,
                    "actual_arrival": revised,
                    "flight_number": entry["number"],
                    "airline": entry["airline"]["name"],
                    "status": entry["status"]
                })
        else:
            print(f"⚠️ request failed for {icao} ({start} to {end}) - status {response.status_code}: {response.text}")

    return pd.DataFrame(records)

In [36]:
berlin_full_day = get_arrivals_for_airport("EDDB", tomorrow) # Berlin only
berlin_full_day

,icao,origin,scheduled_arrival,actual_arrival,flight_number,airline,status
0,EDDB,İzmir,2026-09-25 06:05+02:00,2026-09-25 06:05+02:00,XQ 966,Sun Express,Expected
1,EDDB,Gaziantep,2026-09-25 06:45+02:00,2026-09-25 06:45+02:00,XQ 1766,Sun Express,Expected
2,EDDB,Beijing,2026-09-25 06:45+02:00,2026-09-25 06:45+02:00,HU 489,Hainan,Expected
3,EDDB,Bucharest,2026-09-25 07:05+02:00,2026-09-25 07:05+02:00,W4 3109,Aero Services Executive,Expected
4,EDDB,Bratislava,2026-09-25 07:15+02:00,2026-09-25 07:15+02:00,W6 7037,Wizz Air,Expected
...,...,...,...,...,...,...,...
520,EDDB,Helsinki,2026-09-25 23:00+02:00,2026-09-25 23:00+02:00,EW 8231,Eurowings,Expected
521,EDDB,Burgas,2026-09-25 23:05+02:00,2026-09-25 23:05+02:00,SR 7359,SundAir,Expected
522,EDDB,Kos Island,2026-09-25 23:05+02:00,2026-09-25 23:05+02:00,A3 3966,Aegean,Expected
523,EDDB,Kos Island,2026-09-25 23:05+02:00,2026-09-25 23:05+02:00,EW 8671,Eurowings,Expected


### Automating Arrivals for All Airports

Consolidating the single-airport function into one that loops through 
all major airports and combines everything into one dataframe. Each 
row gets tagged with city_id and city, so this can link back to 
cities in mysql the same way populations and forecasts do.

In [37]:
def get_all_arrivals(major_airports_df: pd.DataFrame, day: str) -> pd.DataFrame:
    """
    Loops through each airport in major_airports_df and gets a full
    day's arrivals for it, tagging each row with city_id and city so
    it can be linked back to the cities table later.
    """
    all_arrivals = []

    for _, row in major_airports_df.iterrows():
        icao = row["icao"]
        city_id = row["city_id"]
        city = row["city"]

        # reusing the single-airport function built earlier - this
        # already handles the two 12-hour windows and the status code check
        airport_arrivals = get_arrivals_for_airport(icao, day)

        if not airport_arrivals.empty:
            airport_arrivals["city_id"] = city_id
            airport_arrivals["city"] = city
            all_arrivals.append(airport_arrivals)
        else:
            print(f"⚠️ no arrivals data returned for {city} ({icao})")

    if not all_arrivals:
        raise ValueError("No arrivals data was returned for any airport - check the warnings above")

    return pd.concat(all_arrivals, ignore_index=True)

In [38]:
arrivals_df = get_all_arrivals(major_airports_df, tomorrow)
arrivals_df

,icao,origin,scheduled_arrival,actual_arrival,flight_number,airline,status,city_id,city
0,EDDB,İzmir,2026-09-25 06:05+02:00,2026-09-25 06:05+02:00,XQ 966,Sun Express,Expected,1,Berlin
1,EDDB,Gaziantep,2026-09-25 06:45+02:00,2026-09-25 06:45+02:00,XQ 1766,Sun Express,Expected,1,Berlin
2,EDDB,Beijing,2026-09-25 06:45+02:00,2026-09-25 06:45+02:00,HU 489,Hainan,Expected,1,Berlin
3,EDDB,Bucharest,2026-09-25 07:05+02:00,2026-09-25 07:05+02:00,W4 3109,Aero Services Executive,Expected,1,Berlin
4,EDDB,Newark,2026-09-25 07:15+02:00,2026-09-25 07:15+02:00,UA 962,United,Expected,1,Berlin
...,...,...,...,...,...,...,...,...,...
1225,EDDM,Gran Canaria Island,2026-09-25 23:15+02:00,2026-09-25 23:15+02:00,DE 1523,Condor,Expected,3,Munich
1226,EDDM,Lisbon,2026-09-25 23:20+02:00,2026-09-25 23:20+02:00,TP 556,TAP Air Portugal,Expected,3,Munich
1227,EDDM,Palma De Mallorca,2026-09-25 23:20+02:00,2026-09-25 23:20+02:00,4Y 453,Eurowings Discover,Expected,3,Munich
1228,EDDM,London,2026-09-25 23:20+02:00,2026-09-25 23:20+02:00,LH 2481,Lufthansa,Expected,3,Munich


In [39]:
arrivals_df["city"].value_counts()

city
Berlin     525
Munich     514
Hamburg    191
Name: count, dtype: int64

## 14. Loading Flight Data into MySQL

### Creating the Airports and Arrivals Tables

airports is static-ish info, similar to cities - one row per city's 
major airport. arrivals is time-based data, same pattern as 
populations and forecasts. Both link back to cities via city_id. 
Dropping and recreating both tables so re-running this notebook never 
creates duplicates - arrivals and airports both need to be dropped 
before cities, since both point back to it.

In [40]:
# dropping arrivals and airports before cities, since both have a
# foreign key pointing back to cities - same reason forecasts had to
# be dropped before cities earlier
with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS arrivals"))
    conn.execute(text("DROP TABLE IF EXISTS airports"))
    conn.commit()

create_airports_table = """
CREATE TABLE airports (
    airport_id INT PRIMARY KEY AUTO_INCREMENT,
    city_id INT NOT NULL,
    icao VARCHAR(10) NOT NULL,
    iata VARCHAR(10),
    name VARCHAR(255),
    FOREIGN KEY (city_id) REFERENCES cities(city_id)
);
"""

create_arrivals_table = """
CREATE TABLE arrivals (
    arrival_id INT PRIMARY KEY AUTO_INCREMENT,
    city_id INT NOT NULL,
    icao VARCHAR(10) NOT NULL,
    origin VARCHAR(255),
    scheduled_arrival VARCHAR(50),
    actual_arrival VARCHAR(50),
    flight_number VARCHAR(20),
    airline VARCHAR(255),
    status VARCHAR(50),
    FOREIGN KEY (city_id) REFERENCES cities(city_id)
);
"""

with engine.connect() as conn:
    conn.execute(text(create_airports_table))
    conn.execute(text(create_arrivals_table))
    conn.commit()

### Inserting with .to_sql()

Same approach as forecasts - loading both dataframes into their 
tables in one line each, instead of writing insert loops manually.

In [41]:
# airports table only needs city_id, icao, iata, name - trimming
# major_airports_df down to just those columns before inserting
airports_to_insert = major_airports_df[["city_id", "icao", "iata", "name"]]

airports_to_insert.to_sql(
    name="airports",
    con=engine,
    if_exists="append",
    index=False
)

# arrivals_df already has all the right columns except "city" (text),
# which isn't needed once city_id is there - dropping it before insert
arrivals_to_insert = arrivals_df.drop(columns=["city"])

arrivals_to_insert.to_sql(
    name="arrivals",
    con=engine,
    if_exists="append",
    index=False
)

1230

### Checking It Worked

In [42]:
display(pd.read_sql("SELECT * FROM airports", engine))
display(pd.read_sql("SELECT city_id, COUNT(*) AS row_count FROM arrivals GROUP BY city_id", engine))

,airport_id,city_id,icao,iata,name
0,1,1,EDDB,BER,Berlin Brandenburg
1,2,2,EDDH,HAM,Hamburg
2,3,3,EDDM,MUC,Munich


,city_id,row_count
0,1,525
1,2,191
2,3,514
